# Lesson 01 · The beta distribution

**Source:** *Introduction to Empirical Bayes: Examples from Baseball Statistics*, David Robinson (2017), Chapter 2 (pp. 12–19)

**Question:** A batter's first at-bat is a hit, so his average is 1.000. Why should that barely change what we believe about him, and how do we put that belief into numbers?

**Goal:** Use the beta distribution as a distribution *of probabilities*, update it with hits and misses, and check by simulation that the update rule really is Bayes' theorem.

**Contents**

1. The beta distribution's shapes
2. A prior for batting averages
3. Updating: prior + evidence = posterior
4. Why the update works: simulating ten million players
5. Exercises (including a card-pack version)

**Tags:** `beta-distribution` `binomial` `prior` `posterior` `conjugate-prior` `simulation`

**Before you start:** No data needed. Read Chapter 2 of the book first, or alongside.

*How this notebook works:* each **Your turn** prompt is followed by an empty cell for your code. **Check** boxes give values to compare against; they come from the book, and your numbers can differ slightly because the Lahman data now runs through more recent seasons. **Reflect** prompts are for short written answers.

In [ ]:
# --- Setup: run this cell first ------------------------------------------
# Windows + conda: put the environment's DLL folders on PATH, so plotting
# can't crash the kernel when Jupyter wasn't started from the activated env.
# (If ENVIRONMENT_NOTES.md in the Cardprice folder has a PATH-fix block,
# you can paste that here instead.)
import os, sys
for sub in ["Library\\mingw-w64\\bin", "Library\\usr\\bin", "Library\\bin", "Scripts", "bin"]:
    p = os.path.join(sys.prefix, sub)
    if os.path.isdir(p) and p not in os.environ["PATH"]:
        os.environ["PATH"] = p + os.pathsep + os.environ["PATH"]

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy
from scipy import stats, optimize, special

import eb_data   # helpers in this folder: loading the Lahman data, saving results

rng = np.random.default_rng(2017)
pd.set_option("display.precision", 4)
plt.rcParams["figure.figsize"] = (7, 4)
print(f"numpy {np.__version__} | pandas {pd.__version__} | scipy {scipy.__version__}")

## 1. The beta distribution's shapes

The beta distribution lives on the interval (0, 1), so each value it produces can be read as a probability. It has two shape parameters, $\alpha$ and $\beta$:

$$\text{mean} = \frac{\alpha}{\alpha+\beta}, \qquad \text{variance} = \frac{\alpha\beta}{(\alpha+\beta)^2(\alpha+\beta+1)}.$$

A handy way to think about it: $\alpha$ counts "successes", $\beta$ counts "failures", and $\alpha+\beta$ says how much evidence the curve represents.

**Your turn.** Plot the densities of Beta(1, 2), Beta(3, 3), Beta(20, 20) and Beta(50, 10) on one chart over (0, 1). Use `stats.beta(a, b).pdf(x)`.

In [ ]:
# Your code here

**Your turn.** For each of the four, compute the mean and standard deviation, both from the formulas above and with `stats.beta(a, b).mean()` / `.std()`. Put them in a small DataFrame.

In [ ]:
# Your code here

**Reflect.** How do $\alpha$ and $\beta$ control the *location* of the curve, and how does their *sum* control its width? Compare Beta(3, 3) with Beta(20, 20).

*Your answer:*

## 2. A prior for batting averages

Before a player's first at-bat, we already know a lot: across baseball history, most season-long averages fall roughly between .210 and .350, with .270 typical. The book encodes that knowledge as Beta(81, 219).

**Your turn.** Plot Beta(81, 219) over (0, 0.5). Compute its mean and the central 95% range (`.ppf([0.025, 0.975])`).

In [ ]:
# Your code here

> **Check:** mean ≈ 0.270; the 95% range is roughly 0.22 to 0.32.

**Your turn.** Where did 81 and 219 come from? Work backwards (the *method of moments*): choose a mean $\mu = 0.27$ and a standard deviation you think fits "mostly between .21 and .35", then solve for $\alpha$ and $\beta$. Hint: with $\nu = \alpha+\beta$, the variance is $\mu(1-\mu)/(\nu+1)$.

In [ ]:
# Your code here

## 3. Updating: prior + evidence = posterior

If the prior is Beta($\alpha_0$, $\beta_0$) and a player gets $H$ hits in $AB$ at-bats, the posterior is

$$\text{Beta}(\alpha_0 + H,\; \beta_0 + AB - H).$$

That is the whole update: add hits to $\alpha$ and misses to $\beta$.

**Your turn.** On one chart, plot the prior Beta(81, 219), the posterior after 1 hit in 1 at-bat, and the posterior after 100 hits in 300 at-bats.

In [ ]:
# Your code here

**Your turn.** Compute the posterior mean after 100/300. Compare it with the raw average (100/300) and the prior mean (81/300).

In [ ]:
# Your code here

> **Check:** the posterior mean is about 0.30: between the raw .333 and the prior .270.

**Reflect.** The book describes the prior as giving every player a *head start* of hits and misses. How many at-bats' worth of head start does Beta(81, 219) give? After how many real at-bats does the player's own record outweigh the prior?

*Your answer:*

## 4. Why the update works: simulating ten million players

Here's the update without any algebra. Simulate a huge population of players whose true averages come from the prior, let each bat 300 times, then keep only the ones who got exactly 100 hits. Their true averages make up the posterior.

Memory tip: 10 million draws is about 80 MB per array. Start with 1 million if your laptop complains.

**Your turn.** Simulate `true_average = rng.beta(81, 219, size=N)` and `hits = rng.binomial(300, true_average)`. Keep both in a DataFrame.

In [ ]:
# Your code here

**Your turn.** Filter to players with exactly 100 hits. Plot a histogram (density scale) of their true averages, and overlay the Beta(81 + 100, 219 + 200) density.

In [ ]:
# Your code here

> **Check:** the histogram and the curve should match closely, and the median of the filtered true averages should be about 0.30.

**Your turn.** Repeat for players with 60, 80 and 100 hits, and plot the three densities together (a KDE via `stats.gaussian_kde`, or overlaid histograms).

In [ ]:
# Your code here

**Reflect.** In one or two sentences in your own words: what question is Bayesian updating answering when it combines a prior with evidence?

*Your answer:*

## 5. Exercises

**Your turn.** A player goes 0 for 10. What's his posterior mean under the Beta(81, 219) prior? And under a much weaker prior with the same mean, Beta(8.1, 21.9)? Which prior would you trust for a major-league hitter, and why?

In [ ]:
# Your code here

**Your turn.** **Card connection.** You open packs of a new set and want to estimate the chance that a pack contains a special illustration rare. From earlier sets you believe the rate is about 1 in 20 packs, but it could plausibly be anywhere from 1 in 40 to 1 in 10. Pick a beta prior to match, then update it after opening 12 packs and finding 2 hits. Plot the prior and posterior.

In [ ]:
# Your code here

## References

- *Introduction to Empirical Bayes: Examples from Baseball Statistics*, David Robinson (2017), Chapter 2.
- David Robinson's original Cross Validated answer, "What is the intuition behind beta distribution?": https://stats.stackexchange.com/questions/47771
- McElreath, *Statistical Rethinking* (2nd ed.), Chapter 2: the same updating idea, done with grid approximation.